# Smoke Then Pilot Workflow

Use this notebook in Google Colab Pro. Run the smoke workflow first, and only continue to pilot after both smoke training runs finish cleanly.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
from pathlib import Path

REPO_ROOT = Path('/content/CS 690L')  # Update if you cloned/uploaded elsewhere.
if not (REPO_ROOT / 'pyproject.toml').exists():
    raise FileNotFoundError(f'Repo checkout not found at {REPO_ROOT}. Update REPO_ROOT before continuing.')
os.chdir(REPO_ROOT)

DRIVE_ROOT = Path('/content/drive/MyDrive/mm-align')
RAW_ROOT = DRIVE_ROOT / 'data' / 'raw'
PROCESSED_ROOT = DRIVE_ROOT / 'data' / 'processed'
ARTIFACTS_ROOT = DRIVE_ROOT / 'artifacts' / 'runs'
HF_ROOT = DRIVE_ROOT / '.hf'

for path in (RAW_ROOT, PROCESSED_ROOT, ARTIFACTS_ROOT, HF_ROOT):
    path.mkdir(parents=True, exist_ok=True)

os.environ['MM_ALIGN_RAW_DIR'] = str(RAW_ROOT)
os.environ['MM_ALIGN_PROCESSED_DIR'] = str(PROCESSED_ROOT)
os.environ['MM_ALIGN_ARTIFACTS_DIR'] = str(ARTIFACTS_ROOT)
os.environ['HF_HOME'] = str(HF_ROOT)
os.environ['HF_DATASETS_CACHE'] = str(HF_ROOT / 'datasets')
os.environ['TRANSFORMERS_CACHE'] = str(HF_ROOT / 'hub')

for key in ['MM_ALIGN_RAW_DIR', 'MM_ALIGN_PROCESSED_DIR', 'MM_ALIGN_ARTIFACTS_DIR', 'HF_HOME', 'HF_DATASETS_CACHE', 'TRANSFORMERS_CACHE']:
    print(f'{key}={os.environ[key]}')


In [ ]:
!bash scripts/colab_setup.sh


In [ ]:
!bash scripts/colab_smoke_run.sh


## Pilot Runs

Continue only after the smoke DPO and smoke image-aware runs both finish successfully.

In [ ]:
!python -m mm_align.cli prepare-data --config configs/pilot.yaml
!python -m mm_align.cli train-dpo --config configs/pilot.yaml


In [ ]:
!python -m mm_align.cli train-imgaware --config configs/pilot.yaml


## Evaluation

Replace `<run_id>` with the emitted run identifier from a completed pilot training run.

In [ ]:
!python -m mm_align.cli evaluate --config configs/pilot.yaml --run <run_id>
!python -m mm_align.cli build-dashboard-data --run <run_id> --artifacts-dir "$MM_ALIGN_ARTIFACTS_DIR"
